In [33]:
#!/home/mchan/Doc/venv_new_qiskit/bin/python3 -u
from mpi4py import MPI
from datetime import datetime
import numpy as np
import copy
import psutil
from scipy.special import ive
import random as rd
import pickle
from scipy import sparse
import gc
from qiskit_nature.second_q.hamiltonians.lattices import (
    BoundaryCondition,
    Lattice,
    LineLattice,
    SquareLattice
)
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.operators import FermionicOp
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.hamiltonians import FermiHubbardModel, QuadraticHamiltonian
from qiskit_nature.second_q.circuit.library import BogoliubovTransform

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

comm = MPI.COMM_WORLD

core = comm.Get_rank()
cores = comm.Get_size()

n_x     = 4
n_y     = 1
n_site  = n_x * n_y
n_qubit = 2*n_site
dim = 2**n_qubit

n_dimer = n_site//2
# For a chain
n_inter = n_dimer 
# For a dimer 
#n_inter = 2*n_dimer


In [34]:
Uc           = 4.0
mu           = Uc/2.0 # half-filling fermi level
n_electrons  = n_site # half-filling
t_hop        = 1.0
t_intra      = t_hop
t_inters     = np.linspace(0.0,1.0,12)
#t_inters     = np.linspace(0.0,1.0,2)
nt_inter     = len(t_inters)
for it_inter in range(nt_inter):
    t_inters[it_inter] *= t_hop
kinetic_parts       = []
hamiltonians        = []
it_inter            = 0
bc = BoundaryCondition.OPEN

# make interaction part first (fixed)

if (n_y==1):
    empty  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=0.0,\
                            onsite_parameter=0.0)
else:
    empty = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(0,0),\
                        onsite_parameter=0.0)
interaction_part = FermiHubbardModel(lattice=empty, onsite_interaction=Uc).second_q_op().simplify()
for t_inter in t_inters:
    if (n_y==1):
        square  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=-t_inter,\
                                onsite_parameter=-mu)
    else:
        square = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(-t_inter,-t_inter),\
                                onsite_parameter=-mu)
    square_modified_graph = square.graph
    # replace intra dimer hoppings
    # assumes dimer direction in x-axis
    for i_dimer in range(n_dimer):
        i_site = 2*i_dimer
        square_modified_graph.update_edge(i_site,i_site+1,-t_intra)
    
    square_modified = Lattice(square_modified_graph)
    
    hopping_matrix = np.zeros((n_qubit,n_qubit),dtype=complex)
    for i_site, j_site, weight in square_modified_graph.weighted_edge_list():
        i_up = 2*i_site
        i_dn = 2*i_site + 1
        j_up = 2*j_site
        j_dn = 2*j_site + 1
        hopping_matrix[i_up,j_up] = weight
        hopping_matrix[i_dn,j_dn] = weight
        hopping_matrix[j_up,i_up] = weight # conjugate actually
        hopping_matrix[j_dn,i_dn] = weight

    kinetic_part = QuadraticHamiltonian(hermitian_part=hopping_matrix)
    kinetic_parts.append(kinetic_part)
    hamiltonian = kinetic_part.second_q_op().simplify() + interaction_part

    # constant_term, to match with references
    # Create the constant term as a FermionicOp
    constant_term = FermionicOp({'': 
        0.25 * Uc * n_site
    })

    hamiltonian += constant_term

    mapper = JordanWignerMapper()
    hamiltonians.append(mapper.map(hamiltonian))
    if (core==0):
        print(it_inter,hamiltonians[it_inter])
    it_inter += 1


0 SparsePauliOp(['IIIIIYZY', 'IIIIIXZX', 'IIIIYZYI', 'IIIIXZXI', 'IYZYIIII', 'IXZXIIII', 'YZYIIIII', 'XZXIIIII', 'ZZIIIIII', 'IIIIIIZZ', 'IIIIZZII', 'IIZZIIII'],
              coeffs=[-0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j,
 -0.5+0.j,  1. +0.j,  1. +0.j,  1. +0.j,  1. +0.j])
1 SparsePauliOp(['IIIIIYZY', 'IIIIIXZX', 'IIIIYZYI', 'IIIIXZXI', 'IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII', 'IYZYIIII', 'IXZXIIII', 'YZYIIIII', 'XZXIIIII', 'ZZIIIIII', 'IIIIIIZZ', 'IIIIZZII', 'IIZZIIII'],
              coeffs=[-0.5       +0.j, -0.5       +0.j, -0.5       +0.j, -0.5       +0.j,
 -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j,
 -0.5       +0.j, -0.5       +0.j, -0.5       +0.j, -0.5       +0.j,
  1.        +0.j,  1.        +0.j,  1.        +0.j,  1.        +0.j])
2 SparsePauliOp(['IIIIIYZY', 'IIIIIXZX', 'IIIIYZYI', 'IIIIXZXI', 'IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII', 'IYZYIIII', 'IXZXIIII', 'YZYIIIII', 'XZXIIIII', 'ZZIIIIII', 'IIIIIIZZ', '

In [35]:
#Uc_max           = 4.0 # 10.0 # 4.0
#n_electrons  = n_site # half-filling
#t_hop        = 1.0
#t_intra      = t_hop
##t_inters     = np.linspace(0.0,1.0,5)
#Ucs          = np.linspace(0.0,1.0,3) * Uc_max
#kinetic_parts       = []
#hamiltonians        = []
#i_U = 0
#bc = BoundaryCondition.OPEN
#
#for Uc in Ucs:
#    mu           = Uc/2.0 # half-filling fermi level
#    if (n_y==1):
#        square  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=-t_inter,\
#                                onsite_parameter=-mu)
#    else:
#        square = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(-t_inter,-t_inter),\
#                                onsite_parameter=-mu)
#    if (n_y==1):
#        empty  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=0.0,\
#                                onsite_parameter=0.0)
#    else:
#        empty = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(0,0),\
#                            onsite_parameter=0.0)
#    interaction_part = FermiHubbardModel(lattice=empty, onsite_interaction=Uc).second_q_op().simplify()
#
#    square_modified_graph = square.graph
#    
#    hopping_matrix = np.zeros((n_qubit,n_qubit),dtype=complex)
#    for i_site, j_site, weight in square_modified_graph.weighted_edge_list():
#        i_up = 2*i_site
#        i_dn = 2*i_site + 1
#        j_up = 2*j_site
#        j_dn = 2*j_site + 1
#        hopping_matrix[i_up,j_up] = weight
#        hopping_matrix[i_dn,j_dn] = weight
#        hopping_matrix[j_up,i_up] = weight # conjugate actually
#        hopping_matrix[j_dn,i_dn] = weight
#
#    kinetic_part = QuadraticHamiltonian(hermitian_part=hopping_matrix)
#    kinetic_parts.append(kinetic_part)
#    hamiltonian = kinetic_part.second_q_op().simplify() + interaction_part
#
#    # constant_term, to match with references
#    # Create the constant term as a FermionicOp
#    constant_term = FermionicOp({'': 
#        0.25 * Uc * n_site
#    })
#
#    hamiltonian += constant_term
#
#    mapper = JordanWignerMapper()
#    hamiltonians.append(mapper.map(hamiltonian))
#    if (core==0):
#        print(i_U,hamiltonians[i_U])
#    i_U += 1
#

In [36]:
n_hamiltonians = len(hamiltonians)

In [37]:
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))


# Hamiltonian differences
0 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
1 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
2 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
3 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
4 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
5 SparsePauliOp(['IIIYZYII', 'IIIXZXII', 'IIYZYIII', 'IIXZXIII'],              coeffs=[-0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j, -0.04545455+0.j])
6 SparsePauliOp(['IIIYZYII', 'IIIXZXII

In [38]:
if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

# Hamiltonian differences_list
[('IIIYZYII', (-0.045454545454545456+0j)), ('IIIXZXII', (-0.045454545454545456+0j)), ('IIYZYIII', (-0.045454545454545456+0j)), ('IIXZXIII', (-0.045454545454545456+0j))]
[('IIIYZYII', (-0.045454545454545456+0j)), ('IIIXZXII', (-0.045454545454545456+0j)), ('IIYZYIII', (-0.045454545454545456+0j)), ('IIXZXIII', (-0.045454545454545456+0j))]
[('IIIYZYII', (-0.04545454545454544+0j)), ('IIIXZXII', (-0.04545454545454544+0j)), ('IIYZYIII', (-0.04545454545454544+0j)), ('IIXZXIII', (-0.04545454545454544+0j))]
[('IIIYZYII', (-0.04545454545454547+0j)), ('IIIXZXII', (-0.04545454545454547+0j)), ('IIYZYIII', (-0.04545454545454547+0j)), ('IIXZXIII', (-0.04545454545454547+0j))]
[('IIIYZYII', (-0.04545454545454547+0j)), ('IIIXZXII', (-0.04545454545454547+0j)), ('IIYZYIII', (-0.04545454545454547+0j)), ('IIXZXIII', (-0.04545454545454547+0j))]
[('IIIYZYII', (-0.045454545454545414+0j)), ('IIIXZXII', (-0.045454545454545414+0j)), ('IIYZYIII', (-0.045454545454545414+0j)), ('IIXZXII

In [39]:
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    factor = 2*2 # 2 for spin, 2 for XX and YY
    pauli_list = ['IIIXZXII']
    coeffs     = np.asarray([factor * -0.5 * (t_inters[alpha+1]-t_inters[alpha])])
    for i_pauli in range(len(pauli_list)):
        for i_qubit in range(n_qubit-len(pauli_list[i_pauli])):
            pauli_list[i_pauli] = 'I' + pauli_list[i_pauli]
    dH = SparsePauliOp(pauli_list, coeffs=coeffs)
    hamiltonian_diffs_reduced.append(dH)
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

0 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
1 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
2 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
3 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
4 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
5 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
6 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
7 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
8 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
9 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
10 SparsePauliOp(['IIIXZXII'],              coeffs=[-0.18181818+0.j])
# Hamiltonian differences_reduced_list
[('IIIXZXII', (-0.18181818181818182+0j))]
[('IIIXZXII', (-0.18181818181818182+0j))]
[('IIIXZXII', (-0.18181818181818177+0j))]
[('IIIXZXII', (-0.18181818181818188+0j))]
[('IIIXZXII', (-0.181818181818181

In [40]:
# sectorize
nsec = 1
dim_sub = [0 for _ in range(nsec)]
indx_sub = [[] for _ in range(nsec)]
iindx_sub = [[] for _ in range(nsec)]

n_up_target = [0 for _ in range(nsec)]
n_dn_target = [0 for _ in range(nsec)]

# consider only half-filled sector
n_up_target[0] = n_site//2
n_dn_target[0] = n_site//2

for isec in range(nsec):
    dim_sub[isec] = 0
    indx_sub[isec] = []

    for i in range(dim):
        s = f'{i:0{n_qubit}b}'
        n_up = 0
        for j in range(n_site):
            n_up += int(s[2*j])
        n_dn = 0
        for j in range(n_site):
            n_dn += int(s[2*j+1])
        if (n_up==n_up_target[isec] and n_dn==n_dn_target[isec]):
            dim_sub[isec] += 1
            indx_sub[isec].append(i)

    if (core==0):
        st = '# dimension of subspace with (n_up,n_dn) = ({n_up},{n_dn}) = {dim_sub}'.format(n_up=n_up_target[isec],n_dn=n_dn_target[isec],dim_sub=dim_sub[isec])
        print(st)
    indx_sub[isec] = np.array(indx_sub[isec])
    # inverse of indx_sub
    iindx_sub[isec] = -np.ones((dim),dtype=int)
    for i in range(dim_sub[isec]):
        iindx_sub[isec][indx_sub[isec][i]] = i


# dimension of subspace with (n_up,n_dn) = (2,2) = 36


In [41]:
#def ApplyParticleHole(i):
#    bit_expr = bin(i)[2:].zfill(n_qubit)
#    operated = ''
#    for k in range(n_qubit):
#        if (bit_expr[k] =='0'):
#            operated += '1'
#        else :
#            operated += '0'
#    return int(operated,2)
#def ApplySpinReverse(i):
#    bit_expr = bin(i)[2:].zfill(n_qubit)
#    operated = ''
#    n_orbital = n_qubit//2
#    for k in range(n_orbital):
#        i_up = 2*k
#        i_dn = 2*k+1
#        operated += bit_expr[i_dn]
#        operated += bit_expr[i_up]
#    return int(operated,2)
#def ApplyOrderReverse(i):
#    bit_expr = bin(i)[2:].zfill(n_qubit)
#    operated = ''
#    for k in reversed(range(n_qubit)):
#        operated += bit_expr[k]
#    return int(operated,2)

In [42]:
#i = 12
#j = ApplyParticleHole(i)
#k = ApplySpinReverse(i)
#l = ApplyOrderReverse(i)
#print(bin(i)[2:].zfill(n_qubit),bin(j)[2:])
#print(bin(k)[2:].zfill(n_qubit),bin(l)[2:].zfill(n_qubit))

In [43]:
# symmetry sectorization
#isec = 0
#done = [False] * dim_sub[isec]
#S = []
#indx_S = 0
#for i in range(dim_sub[isec]):
#    if (done[i]):
#        continue
#    S.append([i])
#    # apply symmetry operation
#    ss = []
#    for l in S[indx_S]:
#        j = ApplyOrderReverse(indx_sub[isec][l])
#        jj = iindx_sub[isec][j]
#        if (jj not in S[indx_S]):
#            ss += [jj]
#            done[jj] = True
#    S[indx_S] += ss
#
#    ss = []
#    for l in S[indx_S]:
#        j = ApplySpinReverse(indx_sub[isec][l])
#        jj = iindx_sub[isec][j]
#        if (jj not in S[indx_S]):
#            ss += [jj]
#            done[jj] = True
#    S[indx_S] += ss
#
#    ss = []
#    for l in S[indx_S]:
#        j = ApplyParticleHole(indx_sub[isec][l])
#        jj = iindx_sub[isec][j]
#        if (jj not in S[indx_S]):
#            ss += [jj]
#            done[jj] = True
#    S[indx_S] += ss
#
#    indx_S += 1
#num_S = len(S)
#print(num_S)
##for j in range(num_S):
##    print([bin(indx_sub[isec][k])[2:].zfill(n_qubit) for k in S[j]])

In [44]:
#def OrderReverseSign(i):
#    return 1
## compose particle hole symmetry operator Ph
#isec = 0
#Or = np.zeros((dim_sub[0],dim_sub[0]),dtype=complex)
#for jj in range(dim_sub[0]):
#    j = indx_sub[isec][jj]
#    i = ApplyOrderReverse(j)
#    ii = iindx_sub[isec][i]
#    Or[ii,jj] = OrderReverseSign(i)
#
#
#def ParticleHoleSign(i):
#    bit_expr = bin(i)[2:].zfill(n_qubit)
#    operated = ''
#    n_orbital = n_qubit//2
#    sign_exp = 0
#    for k in range(n_orbital):
#        i_up = 2*k
#        i_dn = 2*k+1
#        sign_exp += (int(bit_expr[i_up]) + int(bit_expr[i_dn])) * k
#    sign_exp = sign_exp % 2
#    return (-1) ** sign_exp
## compose particle hole symmetry operator Ph
#isec = 0
#Ph = np.zeros((dim_sub[0],dim_sub[0]),dtype=complex)
#for jj in range(dim_sub[0]):
#    j = indx_sub[isec][jj]
#    i = ApplyParticleHole(j)
#    ii = iindx_sub[isec][i]
#    Ph[ii,jj] = ParticleHoleSign(i)
#
#def SpinReverseSign(i):
#    bit_expr = bin(i)[2:].zfill(n_qubit)
#    operated = ''
#    n_orbital = n_qubit//2
#    sign_exp = 0
#    for k in range(n_orbital):
#        i_up = 2*k
#        i_dn = 2*k+1
#        sign_exp += int(bit_expr[i_up]) * int(bit_expr[i_dn])
#    sign_exp = sign_exp % 2
#    return (-1) ** sign_exp
## compose spin reverse symmetry operator Sr
#isec = 0
#Sr = np.zeros((dim_sub[0],dim_sub[0]),dtype=complex)
#for jj in range(dim_sub[0]):
#    j = indx_sub[isec][jj]
#    i = ApplySpinReverse(j)
#    ii = iindx_sub[isec][i]
#    Sr[ii,jj] = SpinReverseSign(i)
#
#

In [45]:
## compose the projection operator
#isec = 0
#identity = np.identity((dim_sub[isec]),dtype=complex)
#projection = (identity + Or) 
#projection = (identity + Ph) @ projection
#if (n_dimer%2==0):
#    projection = (identity + Sr) @ projection
#else:
#    projection = (identity - Sr) @ projection
#projection /= 8

In [46]:
# find the reduced basis by finding eigenstate of projection operator
#basis_transform = np.zeros((num_S,dim_sub[isec]),dtype=complex)
#for indx_S in range(num_S):
#    dim = len(S[indx_S])
#    project_small = np.zeros((dim,dim),dtype=complex)
#    for jjj in range(dim):
#        jj = S[indx_S][jjj]
#        for iii in range(dim):
#            ii = S[indx_S][iii]
#            project_small[iii,jjj] = projection[ii,jj]
#    eigen_e, eigen_v = np.linalg.eigh(project_small)
#    # only eigenvalue=1 (maximum) is valid
#    #print(np.sum(np.abs(eigen_e)>0.5))
#    for jjj in range(dim):
#            jj = S[indx_S][jjj]
#            basis_transform[indx_S,jj] = np.conj(eigen_v[jjj,-1])
##print(basis_transform[0,:])

In [47]:
#eigen_energies_exact   = []
#eigen_vectors_exact   = []
#H_subs = []
#for isec in range(nsec):
#    eigen_energies_exact.append(np.zeros((n_hamiltonians,num_S),dtype=float))
#    eigen_vectors_exact.append(np.zeros((n_hamiltonians,num_S,num_S),dtype=complex))
#    H_subs.append([])
#
#for isec in range(nsec):
#    eigen_e               = np.zeros((num_S),dtype=float)
#    eigen_v               = np.zeros((num_S,num_S),dtype=complex)
#    for alpha in range(n_hamiltonians):
#        start_time = datetime.now()
#        # project hamiltonian on to specified sector
#        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
#        H_sparse.eliminate_zeros()
#        jsec = isec
#        row      = []
#        col      = []
#        data     = []
#        for ii in range(dim_sub[isec]):
#            i = indx_sub[isec][ii]
#            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
#                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
#                #print(i,j)
#                j = H_sparse.indices[ind]
#                jj = iindx_sub[jsec][j]
#                row.append(jj)
#                col.append(ii)
#                #print(ii,jj)
#                data.append(H_sparse.data[ind])
#        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))
#
#        H_subs[isec].append(H_sub)
#
#        # diagonalize sectorized hamiltonian
#        H_projected = basis_transform@(H_sub.toarray())@basis_transform.conj().T
#        #H_projected = projection@(H_sub.toarray())@projection.conj().T
#        eigen_e, eigen_v = np.linalg.eigh(H_projected)
#        #if (isec==nsec-1):
#        print(alpha,eigen_e[0])
#        #print(alpha, eigen_e[0],eigen_e[1] )
#        #    if (alpha==6):
#        #        for k in range(dim_sub[isec]):
#        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
#        #            print(k,np.abs(overlap)**2,eigen_e[k])
#        #
#        #eigen_energies_exact[isec][alpha,:]   = eigen_e
#        #eigen_vectors_exact[isec][alpha,:,:] = eigen_v
#        #end_time = datetime.now()
#        #elapsed = end_time-start_time
#        #elapsed = elapsed.total_seconds()
#        #if (core==0):
#        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
#        #    memory_usage(st)
#

In [48]:
eigen_energies_exact   = []
eigen_vectors_exact   = []
H_subs = []
for isec in range(nsec):
    eigen_energies_exact.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
    eigen_vectors_exact.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))
    H_subs.append([])

for isec in range(nsec):
    eigen_e               = np.zeros(dim_sub[isec],dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    #for alpha in range(n_hamiltonians):
    for alpha in [0, -1]:
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        H_subs[isec].append(H_sub)

        # diagonalize sectorized hamiltonian
        H_projected = H_sub.toarray()
        eigen_e, eigen_v = np.linalg.eigh(H_projected)
        #if (isec==nsec-1):
        print(alpha,eigen_e[0])
        #print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_exact[isec][alpha,:]   = eigen_e
        eigen_vectors_exact[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        if (core==0):
            st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
            memory_usage(st)


0 -5.6568542494923815
[# 8.333333333333332%, elapsed time = 0.003976 secs] memory usage:    0.26830 GiB
-1 -5.9531453086845545
[# 0.0%, elapsed time = 0.004365 secs] memory usage:    0.26830 GiB


In [49]:
# sectorize even more
# is not possible
#alive_0 =  []
#for i in range(dim_sub[0]):
#    indicator = np.abs(eigen_vectors_exact[0][-1,i,0])
#    if (indicator>1e-10):
#        alive_0.append(i)
#        print(indicator)
#print(alive_0)
##sector = [indx_sub[0][0]]
##print(sector)

In [50]:
def ExactEvolution (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_exact[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol

def ExactGaussian (isec, alpha, eps, beta):
    Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=float)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-0.5 * beta ** 2 * (eigen_energies_exact[isec][alpha,k]-eps)**2)
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol


In [51]:
# kinetic parts
hamiltonians_kinetic = []
for alpha in range(n_hamiltonians):
    h = kinetic_parts[alpha].second_q_op().simplify()
    hk = mapper.map(h)
    hamiltonians_kinetic.append(hk)
    if (core==0):
        print(hamiltonians_kinetic[alpha])
# interacting-parts
constant_term = FermionicOp({'': 
    0.25 * Uc * n_site
})
hamiltonian_coulomb = mapper.map(interaction_part+constant_term)
if (core==0):
    print(hamiltonian_coulomb)


SparsePauliOp(['IIIIIIII', 'IIIIIIIZ', 'IIIIIYZY', 'IIIIIXZX', 'IIIIIIZI', 'IIIIYZYI', 'IIIIXZXI', 'IIIIIZII', 'IIIIZIII', 'IIIZIIII', 'IYZYIIII', 'IXZXIIII', 'IIZIIIII', 'YZYIIIII', 'XZXIIIII', 'IZIIIIII', 'ZIIIIIII'],
              coeffs=[-8. +0.j,  1. +0.j, -0.5+0.j, -0.5+0.j,  1. +0.j, -0.5+0.j, -0.5+0.j,
  1. +0.j,  1. +0.j,  1. +0.j, -0.5+0.j, -0.5+0.j,  1. +0.j, -0.5+0.j,
 -0.5+0.j,  1. +0.j,  1. +0.j])
SparsePauliOp(['IIIIIIII', 'IIIIIIIZ', 'IIIIIYZY', 'IIIIIXZX', 'IIIIIIZI', 'IIIIYZYI', 'IIIIXZXI', 'IIIIIZII', 'IIIYZYII', 'IIIXZXII', 'IIIIZIII', 'IIYZYIII', 'IIXZXIII', 'IIIZIIII', 'IYZYIIII', 'IXZXIIII', 'IIZIIIII', 'YZYIIIII', 'XZXIIIII', 'IZIIIIII', 'ZIIIIIII'],
              coeffs=[-8.        +0.j,  1.        +0.j, -0.5       +0.j, -0.5       +0.j,
  1.        +0.j, -0.5       +0.j, -0.5       +0.j,  1.        +0.j,
 -0.04545455+0.j, -0.04545455+0.j,  1.        +0.j, -0.04545455+0.j,
 -0.04545455+0.j,  1.        +0.j, -0.5       +0.j, -0.5       +0.j,
  1.        +0.j, -0

In [52]:
# exact eigenvalues for interaction parts
eigen_energies_coulomb  = []
eigen_vectors_coulomb  = []
for isec in range(nsec):
    eigen_energies_coulomb.append(np.zeros((dim_sub[isec]),dtype=float))
    eigen_vectors_coulomb.append(np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    start_time = datetime.now()
    # project hamiltonian on to specified sector
    H_sparse = hamiltonian_coulomb.to_matrix(sparse=True)
    H_sparse.eliminate_zeros()
    jsec = isec
    row      = []
    col      = []
    data     = []
    for ii in range(dim_sub[isec]):
        i = indx_sub[isec][ii]
        for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
            # j is always in indx_sub[isec], because the Hamiltonian does not mix it
            #print(i,j)
            j = H_sparse.indices[ind]
            jj = iindx_sub[jsec][j]
            row.append(jj)
            col.append(ii)
            #print(ii,jj)
            data.append(H_sparse.data[ind])
    H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

    # diagonalize sectorized hamiltonian
    eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
    #if (isec==nsec-1):
    #print(eigen_e[0],eigen_e[1] )
    #    if (alpha==6):
    #        for k in range(dim_sub[isec]):
    #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
    #            print(k,np.abs(overlap)**2,eigen_e[k])
    #
    eigen_energies_coulomb[isec][:]   = eigen_e
    eigen_vectors_coulomb[isec][:,:] = eigen_v
    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    #if (core==0):
    #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
    #    memory_usage(st)


In [53]:
# exact eigenvalues for kinetic parts
eigen_energies_kinetic   = []
eigen_vectors_kinetic = []
for isec in range(nsec):
    eigen_energies_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
    eigen_vectors_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))

for isec in range(nsec):
    eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
    eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    for alpha in range(n_hamiltonians):
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonians_kinetic[alpha].to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        #print(alpha, eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_kinetic[isec][alpha,:]   = eigen_e
        eigen_vectors_kinetic[isec][alpha,:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)


In [54]:
# prepare sectored pauli (project it)
isec = 0
sectored_pauli = [[None for _ in range(n_hamiltonians)] for _ in range(nsec)]
for alpha in range(n_hamiltonians-1):
    nhd = len(hamiltonian_diffs_reduced_list[alpha])
    sectored_pauli[isec][alpha] = [None for _ in range(nhd)]
    for ihd in range(nhd):
        pauli = hamiltonian_diffs_reduced_list[alpha][ihd][0]
        pauli_op = SparsePauliOp(pauli)
        pauli_sparse = pauli_op.to_matrix(sparse=True)
        pauli_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(pauli_sparse.indptr[i],pauli_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = pauli_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                # project it to isec subspace
                if (jj<0):
                    continue
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(pauli_sparse.data[ind])
        pauli_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))
        sectored_pauli[isec][alpha][ihd]=pauli_sub.toarray()
        #print(sectored_pauli[isec][alpha][ihd])


In [55]:
def ExactEvolution_coulomb (isec, eps, time):
    Vl = copy.deepcopy(eigen_vectors_coulomb[isec][:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_coulomb[isec][k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol


In [56]:
def ExactEvolution_kinetic (isec, alpha, eps, time):
    Vl = copy.deepcopy(eigen_vectors_kinetic[isec][alpha,:,:])
    evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
    vec = np.zeros((dim_sub[isec]),dtype=complex)
    for k in range(dim_sub[isec]):
        vec[k] = np.exp(-1j*time*(eigen_energies_kinetic[isec][alpha,k]-eps))
    exp_d = np.diag(vec)
    evol = Vl@exp_d@Vl.conj().T
    return evol


In [57]:
def TrotterEvolution(isec, alpha, time, eps, n_trotter, indx):
    dt = time/n_trotter
    if (indx==0): 
        # first order trotter
        u_trotter = np.identity((dim_sub[isec]),dtype=complex)
        for i_trotter in range(n_trotter):
            u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
        # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)
        # u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
        # for i_trotter in range(n_trotter-1):
        #     u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
        #     u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
        # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)@u_trotter
    elif (indx==1):
        u_trotter = np.identity((dim_sub[isec]),dtype=complex)
        for i_trotter in range(n_trotter):
            u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
            u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
        #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)
        #u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
        #for i_trotter in range(n_trotter-1):
        #    u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
        #    u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
        #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)@u_trotter
    return u_trotter*np.exp(1j*eps*time)



In [58]:
#isec = 0
#phi_start = eigen_vectors_kinetic[isec][-1,:,0] # kinetic part solution (Hartree Fock)
#eta = np.abs(phi_start.conj().T@eigen_vectors_exact[isec][-1,:,0])**2
#print(eta)

In [59]:
# use pre-computed C99
C99 = 6.3959839

In [60]:
# isec = 0
# phi_start = eigen_vectors_kinetic[isec][-1,:,0] # kinetic part solution (Hartree Fock)
# eta = np.abs(phi_start.conj().T@eigen_vectors_exact[isec][-1,:,0])**2
# construction of starting vector that can be easily constructed within a quantum cirucit
eta_0 = 0.4570251 # gives eta = 0.4000000 for U=4, 4x1
isec = 0
phi_start = np.zeros((dim_sub[isec]),dtype=complex)
theta  = 2.0 * np.arctan(-0.5/t_hop*(Uc/2 + np.sqrt(Uc**2/4+4*t_hop**2)))
# make it different
theta -=    2.0 * np.arccos(eta_0**(1.0/n_site))
basis_index_dimer = [3, 6, 9, 12]
values_dimer      = [np.cos(theta/2.0), np.sin(theta/2.0), -np.sin(theta/2.0), np.cos(theta/2.0)]
basis_indices = []
values        = []
dim_basis = 4**n_dimer
#normsum = 0.0
for i_basis in range(dim_basis):
    indx = i_basis
    ind = 0
    ss  = ''
    value = (1.0/2.0)**(n_dimer/2)
    for i_dimer in range(n_dimer):
        jj = indx%4
        indx = indx// 4
        ind += basis_index_dimer[jj] * 16**i_dimer
        ss = str(jj) + ss
        value = value * values_dimer[jj]
    basis_indices.append(ind)
    values.append(value)
    phi_start[iindx_sub[isec][ind]] = value
    if (core==0):
        print(jj, ind)
    #normsum += value**2
#if (core==0):
#    print('normsum=',normsum)

eta = np.abs(phi_start@eigen_vectors_exact[isec][-1,:,0])**2
if (core==0):
    print(eta)

0 51
0 54
0 57
0 60
1 99
1 102
1 105
1 108
2 147
2 150
2 153
2 156
3 195
3 198
3 201
3 204
0.4000000395498957


In [61]:
def compute_amplitude_samples(amplitude, n_shot):
    # real part
    computed_value = amplitude.real

    # # shot errors
    p_up = (computed_value + 1.0)/2.0
    if (p_up>1.0 or p_up<0.0):
        shot_error = 0.0
    else:
        sample_shot = np.random.binomial(n_shot,p_up)
        shot_error = 2*(sample_shot/(n_shot) - p_up)
    computed_value += shot_error

    amp_sample_real = computed_value
    

    # imag part
    computed_value = amplitude.imag

    # # shot errors
    p_up = (computed_value + 1.0)/2.0
    if (p_up>1.0 or p_up<0.0):
        shot_error = 0.0
    else:
        sample_shot = np.random.binomial(n_shot,p_up)
        shot_error = 2*(sample_shot/(n_shot) - p_up)
    computed_value += shot_error

    amp_sample_imag = computed_value

    amplitude_sample = amp_sample_real + 1j * amp_sample_imag
    return amplitude_sample




In [62]:
# ASP-exact
#n_asp  = 51
#T_asps = np.linspace(0.2, 12, num=n_asp)
# several different dt
dt_list = [0.1, 0.2, 0.5, 1.0]
n_dt = len(dt_list)
T_asps = np.linspace(2.0, 21.0, num=21)
n_asp = len(T_asps)

error_for_dt_list = np.zeros((n_dt,n_asp),dtype=float)
indx_dt = 0
for dt in dt_list:
    E_asps = np.zeros((n_asp))
    F_asps = np.zeros((n_asp))
    for i_asp in range(n_asp):
        T_asp               = T_asps[i_asp]
        nt_inter            = round(T_asps[i_asp]/dt)
        T_asp               = dt * nt_inter
        n_hamiltonians      = nt_inter
        t_inters            = np.linspace(0.0,1.0,num=nt_inter)
        kinetic_parts       = []
        hamiltonians        = []
        it_inter            = 0
        bc = BoundaryCondition.OPEN

        # make interaction part first (fixed)

        if (n_y==1):
            empty  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=0.0,\
                                    onsite_parameter=0.0)
        else:
            empty = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(0,0),\
                                onsite_parameter=0.0)
        interaction_part = FermiHubbardModel(lattice=empty, onsite_interaction=Uc).second_q_op().simplify()
        for t_inter in t_inters:
            if (n_y==1):
                square  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=-t_inter,\
                                        onsite_parameter=-mu)
            else:
                square = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(-t_inter,-t_inter),\
                                        onsite_parameter=-mu)
            square_modified_graph = square.graph
            # replace intra dimer hoppings
            # assumes dimer direction in x-axis
            for i_dimer in range(n_dimer):
                i_site = 2*i_dimer
                square_modified_graph.update_edge(i_site,i_site+1,-t_intra)

            square_modified = Lattice(square_modified_graph)

            hopping_matrix = np.zeros((n_qubit,n_qubit),dtype=complex)
            for i_site, j_site, weight in square_modified_graph.weighted_edge_list():
                i_up = 2*i_site
                i_dn = 2*i_site + 1
                j_up = 2*j_site
                j_dn = 2*j_site + 1
                hopping_matrix[i_up,j_up] = weight
                hopping_matrix[i_dn,j_dn] = weight
                hopping_matrix[j_up,i_up] = weight # conjugate actually
                hopping_matrix[j_dn,i_dn] = weight

            kinetic_part = QuadraticHamiltonian(hermitian_part=hopping_matrix)
            kinetic_parts.append(kinetic_part)
            hamiltonian = kinetic_part.second_q_op().simplify() + interaction_part

            # constant_term, to match with references
            # Create the constant term as a FermionicOp
            constant_term = FermionicOp({'': 
                0.25 * Uc * n_site
            })

            hamiltonian += constant_term

            mapper = JordanWignerMapper()
            hamiltonians.append(mapper.map(hamiltonian))
            #if (core==0):
            #    print(it_inter,hamiltonians[it_inter])
            it_inter += 1


        eigen_energies_exact   = []
        eigen_vectors_exact   = []
        H_subs = []
        for isec in range(nsec):
            eigen_energies_exact.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
            eigen_vectors_exact.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))
            H_subs.append([])

        for isec in range(nsec):
            eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
            eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            for alpha in range(n_hamiltonians):
                start_time = datetime.now()
                # project hamiltonian on to specified sector
                H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
                H_sparse.eliminate_zeros()
                jsec = isec
                row      = []
                col      = []
                data     = []
                for ii in range(dim_sub[isec]):
                    i = indx_sub[isec][ii]
                    for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                        # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                        #print(i,j)
                        j = H_sparse.indices[ind]
                        jj = iindx_sub[jsec][j]
                        row.append(jj)
                        col.append(ii)
                        #print(ii,jj)
                        data.append(H_sparse.data[ind])
                H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

                H_subs[isec].append(H_sub)

                # diagonalize sectorized hamiltonian
                eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
                #if (isec==nsec-1):
                #print(alpha,eigen_e[0])
                #print(alpha, eigen_e[0],eigen_e[1] )
                #    if (alpha==6):
                #        for k in range(dim_sub[isec]):
                #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
                #            print(k,np.abs(overlap)**2,eigen_e[k])
                #
                eigen_energies_exact[isec][alpha,:]   = eigen_e
                eigen_vectors_exact[isec][alpha,:,:] = eigen_v
                end_time = datetime.now()
                elapsed = end_time-start_time
                elapsed = elapsed.total_seconds()
                #if (core==0):
                #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
                #    memory_usage(st)

        def ExactEvolution (isec, alpha, eps, time):
            Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
            evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            vec = np.zeros((dim_sub[isec]),dtype=complex)
            for k in range(dim_sub[isec]):
                vec[k] = np.exp(-1j*time*(eigen_energies_exact[isec][alpha,k]-eps))
            exp_d = np.diag(vec)
            evol = Vl@exp_d@Vl.conj().T
            return evol

        # kinetic parts
        hamiltonians_kinetic = []
        for alpha in range(n_hamiltonians):
            h = kinetic_parts[alpha].second_q_op().simplify()
            hk = mapper.map(h)
            hamiltonians_kinetic.append(hk)
            #if (core==0):
            #    print(hamiltonians_kinetic[alpha])
        # interacting-parts
        constant_term = FermionicOp({'': 
            0.25 * Uc * n_site
        })
        hamiltonian_coulomb = mapper.map(interaction_part+constant_term)
        #if (core==0):
        #    print(hamiltonian_coulomb)

        # exact eigenvalues for interaction parts
        eigen_energies_coulomb  = []
        eigen_vectors_coulomb  = []
        for isec in range(nsec):
            eigen_energies_coulomb.append(np.zeros((dim_sub[isec]),dtype=float))
            eigen_vectors_coulomb.append(np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex))

        for isec in range(nsec):
            eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
            eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            start_time = datetime.now()
            # project hamiltonian on to specified sector
            H_sparse = hamiltonian_coulomb.to_matrix(sparse=True)
            H_sparse.eliminate_zeros()
            jsec = isec
            row      = []
            col      = []
            data     = []
            for ii in range(dim_sub[isec]):
                i = indx_sub[isec][ii]
                for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                    # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                    #print(i,j)
                    j = H_sparse.indices[ind]
                    jj = iindx_sub[jsec][j]
                    row.append(jj)
                    col.append(ii)
                    #print(ii,jj)
                    data.append(H_sparse.data[ind])
            H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

            # diagonalize sectorized hamiltonian
            eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
            #if (isec==nsec-1):
            #print(eigen_e[0],eigen_e[1] )
            #    if (alpha==6):
            #        for k in range(dim_sub[isec]):
            #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
            #            print(k,np.abs(overlap)**2,eigen_e[k])
            #
            eigen_energies_coulomb[isec][:]   = eigen_e
            eigen_vectors_coulomb[isec][:,:] = eigen_v
            end_time = datetime.now()
            elapsed = end_time-start_time
            elapsed = elapsed.total_seconds()
            #if (core==0):
            #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
            #    memory_usage(st)

        # exact eigenvalues for kinetic parts
        eigen_energies_kinetic   = []
        eigen_vectors_kinetic = []
        for isec in range(nsec):
            eigen_energies_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
            eigen_vectors_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))

        for isec in range(nsec):
            eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
            eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            for alpha in range(n_hamiltonians):
                start_time = datetime.now()
                # project hamiltonian on to specified sector
                H_sparse = hamiltonians_kinetic[alpha].to_matrix(sparse=True)
                H_sparse.eliminate_zeros()
                jsec = isec
                row      = []
                col      = []
                data     = []
                for ii in range(dim_sub[isec]):
                    i = indx_sub[isec][ii]
                    for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                        # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                        #print(i,j)
                        j = H_sparse.indices[ind]
                        jj = iindx_sub[jsec][j]
                        row.append(jj)
                        col.append(ii)
                        #print(ii,jj)
                        data.append(H_sparse.data[ind])
                H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

                # diagonalize sectorized hamiltonian
                eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
                #if (isec==nsec-1):
                #print(alpha, eigen_e[0],eigen_e[1] )
                #    if (alpha==6):
                #        for k in range(dim_sub[isec]):
                #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
                #            print(k,np.abs(overlap)**2,eigen_e[k])
                #
                eigen_energies_kinetic[isec][alpha,:]   = eigen_e
                eigen_vectors_kinetic[isec][alpha,:,:] = eigen_v
                end_time = datetime.now()
                elapsed = end_time-start_time
                elapsed = elapsed.total_seconds()
                #if (core==0):
                #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
                #    memory_usage(st)

        def ExactEvolution_coulomb (isec, eps, time):
            Vl = copy.deepcopy(eigen_vectors_coulomb[isec][:,:])
            evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            vec = np.zeros((dim_sub[isec]),dtype=complex)
            for k in range(dim_sub[isec]):
                vec[k] = np.exp(-1j*time*(eigen_energies_coulomb[isec][k]-eps))
            exp_d = np.diag(vec)
            evol = Vl@exp_d@Vl.conj().T
            return evol

        def ExactEvolution_kinetic (isec, alpha, eps, time):
            Vl = copy.deepcopy(eigen_vectors_kinetic[isec][alpha,:,:])
            evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
            vec = np.zeros((dim_sub[isec]),dtype=complex)
            for k in range(dim_sub[isec]):
                vec[k] = np.exp(-1j*time*(eigen_energies_kinetic[isec][alpha,k]-eps))
            exp_d = np.diag(vec)
            evol = Vl@exp_d@Vl.conj().T
            return evol

        def TrotterEvolution(isec, alpha, time, eps, n_trotter, indx):
            dt = time/n_trotter
            if (indx==0): 
                # first order trotter
                u_trotter = np.identity((dim_sub[isec]),dtype=complex)
                for i_trotter in range(n_trotter):
                    u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                    u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)
                # u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                # for i_trotter in range(n_trotter-1):
                #     u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                #     u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)@u_trotter
            elif (indx==1):
                u_trotter = np.identity((dim_sub[isec]),dtype=complex)
                for i_trotter in range(n_trotter):
                    u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                    u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)
                #u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                #for i_trotter in range(n_trotter-1):
                #    u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                #    u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)@u_trotter
            return u_trotter*np.exp(1j*eps*time)

        print('# exact')
        phi = eigen_vectors_exact[0][0,:,0]
        for alpha in range(n_hamiltonians):
            phi = ExactEvolution(0,alpha,0.0,dt)@phi
        E = np.real(phi.conj().T@H_subs[0][-1]@phi)
        fidelity = np.abs(phi.conj().T@eigen_vectors_exact[0][-1,:,0])**2
        #print(T_asp,E,fidelity)
        error = np.abs(E-eigen_energies_exact[0][-1,0])
        #print(T_asp,np.abs(E-eigen_energies_exact[0][-1,0]),fidelity)
        error_for_dt_list[indx_dt,i_asp] = error
        print(dt,T_asp,error)
    indx_dt += 1

    #    n_trotter = 1
    #    indx_trotter = 1
    #    print('# trotter')
    #    phi = eigen_vectors_exact[0][0,:,0]
    #    for alpha in range(n_hamiltonians):
    #        phi = TrotterEvolution(0,alpha,dt,0,n_trotter,indx_trotter)@phi
    #    E = np.real(phi.conj().T@H_subs[0][-1]@phi)
    ##    E = np.real(eigen_vectors_exact[0][-1,:,0].conj().T@H_subs[0][-1]@eigen_vectors_exact[0][-1,:,0])
    #    fidelity = np.abs(phi.conj().T@eigen_vectors_exact[0][-1,:,0])**2
    #    print(T_asp,E,fidelity)
    #    print(T_asp,np.abs(E-eigen_energies_exact[0][-1,0]),fidelity)











# exact
0.1 2.0 0.05829231024368653
# exact
0.1 3.0 0.032453653488969
# exact
0.1 3.9000000000000004 0.01683931083464074
# exact
0.1 4.800000000000001 0.008625316378524595
# exact
0.1 5.800000000000001 0.006702320159726938
# exact
0.1 6.800000000000001 0.004839496915755248
# exact
0.1 7.7 0.004306764046182465
# exact
0.1 8.6 0.0026802734982673826
# exact
0.1 9.600000000000001 0.0022475420304557403
# exact
0.1 10.5 0.0019146119302675402
# exact
0.1 11.5 0.0016259499522126575
# exact
0.1 12.4 0.0014547041218087031
# exact
0.1 13.4 0.0010267182169965139
# exact
0.1 14.4 0.0010922203216612303
# exact
0.1 15.3 0.0008852464987292308
# exact
0.1 16.2 0.0008528616277381218
# exact
0.1 17.2 0.0006683895662300543
# exact
0.1 18.1 0.000603016240250831
# exact
0.1 19.1 0.0006195419469952057
# exact
0.1 20.0 0.0005050059331024315
# exact
0.1 21.0 0.0005027809397422089
# exact
0.2 2.0 0.06096915916807788
# exact
0.2 3.0 0.035925017602674636
# exact
0.2 4.0 0.017000281265490358
# exact
0.2 4.80000000

In [64]:
# save to files
with open('dE_vs_T_for_dt.save','w') as file_:
    s = '# T_asp, dE(dt=0.1), dE(dt=0.2), dE(dt=0.5), dE(dt=0.8)'
    s += '\n'
    file_.write(s)
    for i_asp in range(n_asp):
        s = '{:}'.format(T_asps[i_asp])
        indx_dt = 0 
        for dt in dt_list:
            s += '  {:.16e}'.format(error_for_dt_list[indx_dt,i_asp])
            indx_dt += 1
        print(s)
        s += '\n'
        file_.write(s)

2.0  5.8292310243686529e-02  6.0969159168077880e-02  7.5906031932829521e-02  2.9629105919215348e-01
2.95  3.2453653488969003e-02  3.5925017602674636e-02  4.8571683063330262e-02  1.2600445138855587e-01
3.9  1.6839310834640742e-02  1.7000281265490358e-02  2.0711340413669177e-02  7.5843318414754002e-02
4.85  8.6253163785245945e-03  9.4652504391747883e-03  1.1215685364355465e-02  5.6382338479747496e-02
5.8  6.7023201597269377e-03  6.7923016195381436e-03  7.3767334670504781e-03  3.2959172097786649e-02
6.75  4.8394969157552481e-03  5.0060980373851649e-03  5.5029495647573867e-03  2.6753581124478565e-02
7.699999999999999  4.3067640461824652e-03  4.6516972203374252e-03  5.0515250117539878e-03  1.8897988220776263e-02
8.649999999999999  2.6802734982673826e-03  2.7568897029199846e-03  3.5769119555775220e-03  1.5009479153576422e-02
9.6  2.2475420304557403e-03  2.3591491966294953e-03  2.8287541670817262e-03  1.4242854792493986e-02
10.549999999999999  1.9146119302675402e-03  1.9298798542228823e-03  2

In [52]:
# Trotter
# ASP-exact
dt = 0.5
T_asp = 9.0
max_n_trotter_list = [2, 3, 4, 5, 6, 7, 8, 9, 10, 20, 30, 40, 50, 60]
n_max_n_trotter = len(max_n_trotter_list)
error_for_trotter_list = np.zeros((n_max_n_trotter),dtype=float)
trotter_sum_for_trotter_list = np.zeros((n_max_n_trotter),dtype=float)
indx_max_n_trotter = 0
for max_n_trotter in max_n_trotter_list:
    n_hamiltonians      = nt_inter
    t_inters            = np.linspace(0.0,1.0,num=nt_inter)
    kinetic_parts       = []
    hamiltonians        = []
    it_inter            = 0
    bc = BoundaryCondition.OPEN

    # make interaction part first (fixed)

    if (n_y==1):
        empty  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=0.0,\
                                onsite_parameter=0.0)
    else:
        empty = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(0,0),\
                            onsite_parameter=0.0)
    interaction_part = FermiHubbardModel(lattice=empty, onsite_interaction=Uc).second_q_op().simplify()
    for t_inter in t_inters:
        if (n_y==1):
            square  = LineLattice(num_nodes=n_x,boundary_condition=bc, edge_parameter=-t_inter,\
                                    onsite_parameter=-mu)
        else:
            square = SquareLattice(rows=n_x,cols=n_y, boundary_condition=bc, edge_parameter=(-t_inter,-t_inter),\
                                    onsite_parameter=-mu)
        square_modified_graph = square.graph
        # replace intra dimer hoppings
        # assumes dimer direction in x-axis
        for i_dimer in range(n_dimer):
            i_site = 2*i_dimer
            square_modified_graph.update_edge(i_site,i_site+1,-t_intra)

        square_modified = Lattice(square_modified_graph)

        hopping_matrix = np.zeros((n_qubit,n_qubit),dtype=complex)
        for i_site, j_site, weight in square_modified_graph.weighted_edge_list():
            i_up = 2*i_site
            i_dn = 2*i_site + 1
            j_up = 2*j_site
            j_dn = 2*j_site + 1
            hopping_matrix[i_up,j_up] = weight
            hopping_matrix[i_dn,j_dn] = weight
            hopping_matrix[j_up,i_up] = weight # conjugate actually
            hopping_matrix[j_dn,i_dn] = weight

        kinetic_part = QuadraticHamiltonian(hermitian_part=hopping_matrix)
        kinetic_parts.append(kinetic_part)
        hamiltonian = kinetic_part.second_q_op().simplify() + interaction_part

        # constant_term, to match with references
        # Create the constant term as a FermionicOp
        constant_term = FermionicOp({'': 
            0.25 * Uc * n_site
        })

        hamiltonian += constant_term

        mapper = JordanWignerMapper()
        hamiltonians.append(mapper.map(hamiltonian))
        #if (core==0):
        #    print(it_inter,hamiltonians[it_inter])
        it_inter += 1


    eigen_energies_exact   = []
    eigen_vectors_exact   = []
    H_subs = []
    for isec in range(nsec):
        eigen_energies_exact.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
        eigen_vectors_exact.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))
        H_subs.append([])

    for isec in range(nsec):
        eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
        eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        for alpha in range(n_hamiltonians):
            start_time = datetime.now()
            # project hamiltonian on to specified sector
            H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
            H_sparse.eliminate_zeros()
            jsec = isec
            row      = []
            col      = []
            data     = []
            for ii in range(dim_sub[isec]):
                i = indx_sub[isec][ii]
                for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                    # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                    #print(i,j)
                    j = H_sparse.indices[ind]
                    jj = iindx_sub[jsec][j]
                    row.append(jj)
                    col.append(ii)
                    #print(ii,jj)
                    data.append(H_sparse.data[ind])
            H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

            H_subs[isec].append(H_sub)

            # diagonalize sectorized hamiltonian
            eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
            #if (isec==nsec-1):
            #print(alpha,eigen_e[0])
            #print(alpha, eigen_e[0],eigen_e[1] )
            #    if (alpha==6):
            #        for k in range(dim_sub[isec]):
            #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
            #            print(k,np.abs(overlap)**2,eigen_e[k])
            #
            eigen_energies_exact[isec][alpha,:]   = eigen_e
            eigen_vectors_exact[isec][alpha,:,:] = eigen_v
            end_time = datetime.now()
            elapsed = end_time-start_time
            elapsed = elapsed.total_seconds()
            #if (core==0):
            #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
            #    memory_usage(st)

    def ExactEvolution (isec, alpha, eps, time):
        Vl = copy.deepcopy(eigen_vectors_exact[isec][alpha,:,:])
        evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        vec = np.zeros((dim_sub[isec]),dtype=complex)
        for k in range(dim_sub[isec]):
            vec[k] = np.exp(-1j*time*(eigen_energies_exact[isec][alpha,k]-eps))
        exp_d = np.diag(vec)
        evol = Vl@exp_d@Vl.conj().T
        return evol

    # kinetic parts
    hamiltonians_kinetic = []
    for alpha in range(n_hamiltonians):
        h = kinetic_parts[alpha].second_q_op().simplify()
        hk = mapper.map(h)
        hamiltonians_kinetic.append(hk)
        #if (core==0):
        #    print(hamiltonians_kinetic[alpha])
    # interacting-parts
    constant_term = FermionicOp({'': 
        0.25 * Uc * n_site
    })
    hamiltonian_coulomb = mapper.map(interaction_part+constant_term)
    #if (core==0):
    #    print(hamiltonian_coulomb)

    # exact eigenvalues for interaction parts
    eigen_energies_coulomb  = []
    eigen_vectors_coulomb  = []
    for isec in range(nsec):
        eigen_energies_coulomb.append(np.zeros((dim_sub[isec]),dtype=float))
        eigen_vectors_coulomb.append(np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex))

    for isec in range(nsec):
        eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
        eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        start_time = datetime.now()
        # project hamiltonian on to specified sector
        H_sparse = hamiltonian_coulomb.to_matrix(sparse=True)
        H_sparse.eliminate_zeros()
        jsec = isec
        row      = []
        col      = []
        data     = []
        for ii in range(dim_sub[isec]):
            i = indx_sub[isec][ii]
            for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                #print(i,j)
                j = H_sparse.indices[ind]
                jj = iindx_sub[jsec][j]
                row.append(jj)
                col.append(ii)
                #print(ii,jj)
                data.append(H_sparse.data[ind])
        H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

        # diagonalize sectorized hamiltonian
        eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
        #if (isec==nsec-1):
        #print(eigen_e[0],eigen_e[1] )
        #    if (alpha==6):
        #        for k in range(dim_sub[isec]):
        #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
        #            print(k,np.abs(overlap)**2,eigen_e[k])
        #
        eigen_energies_coulomb[isec][:]   = eigen_e
        eigen_vectors_coulomb[isec][:,:] = eigen_v
        end_time = datetime.now()
        elapsed = end_time-start_time
        elapsed = elapsed.total_seconds()
        #if (core==0):
        #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
        #    memory_usage(st)

    # exact eigenvalues for kinetic parts
    eigen_energies_kinetic   = []
    eigen_vectors_kinetic = []
    for isec in range(nsec):
        eigen_energies_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec]),dtype=float))
        eigen_vectors_kinetic.append(np.zeros((n_hamiltonians,dim_sub[isec],dim_sub[isec]),dtype=complex))

    for isec in range(nsec):
        eigen_e               = np.zeros((dim_sub[isec]),dtype=float)
        eigen_v               = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        for alpha in range(n_hamiltonians):
            start_time = datetime.now()
            # project hamiltonian on to specified sector
            H_sparse = hamiltonians_kinetic[alpha].to_matrix(sparse=True)
            H_sparse.eliminate_zeros()
            jsec = isec
            row      = []
            col      = []
            data     = []
            for ii in range(dim_sub[isec]):
                i = indx_sub[isec][ii]
                for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
                    # j is always in indx_sub[isec], because the Hamiltonian does not mix it
                    #print(i,j)
                    j = H_sparse.indices[ind]
                    jj = iindx_sub[jsec][j]
                    row.append(jj)
                    col.append(ii)
                    #print(ii,jj)
                    data.append(H_sparse.data[ind])
            H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))

            # diagonalize sectorized hamiltonian
            eigen_e, eigen_v = np.linalg.eigh(H_sub.toarray())
            #if (isec==nsec-1):
            #print(alpha, eigen_e[0],eigen_e[1] )
            #    if (alpha==6):
            #        for k in range(dim_sub[isec]):
            #            overlap = eigen_v[:,k].conj()@eigen_vectors_exact[isec][alpha-1,:,0]
            #            print(k,np.abs(overlap)**2,eigen_e[k])
            #
            eigen_energies_kinetic[isec][alpha,:]   = eigen_e
            eigen_vectors_kinetic[isec][alpha,:,:] = eigen_v
            end_time = datetime.now()
            elapsed = end_time-start_time
            elapsed = elapsed.total_seconds()
            #if (core==0):
            #    st = '# {percent}%, elapsed time = {elapsed} secs'.format(percent=((alpha+1)/(nt_inter)*100),elapsed=elapsed)
            #    memory_usage(st)

    def ExactEvolution_coulomb (isec, eps, time):
        Vl = copy.deepcopy(eigen_vectors_coulomb[isec][:,:])
        evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        vec = np.zeros((dim_sub[isec]),dtype=complex)
        for k in range(dim_sub[isec]):
            vec[k] = np.exp(-1j*time*(eigen_energies_coulomb[isec][k]-eps))
        exp_d = np.diag(vec)
        evol = Vl@exp_d@Vl.conj().T
        return evol

    def ExactEvolution_kinetic (isec, alpha, eps, time):
        Vl = copy.deepcopy(eigen_vectors_kinetic[isec][alpha,:,:])
        evol = np.zeros((dim_sub[isec],dim_sub[isec]),dtype=complex)
        vec = np.zeros((dim_sub[isec]),dtype=complex)
        for k in range(dim_sub[isec]):
            vec[k] = np.exp(-1j*time*(eigen_energies_kinetic[isec][alpha,k]-eps))
        exp_d = np.diag(vec)
        evol = Vl@exp_d@Vl.conj().T
        return evol

    def TrotterEvolution(isec, alpha, time, eps, n_trotter, indx):
        dt = time/n_trotter
        if (indx==0): 
            # first order trotter
            u_trotter = np.identity((dim_sub[isec]),dtype=complex)
            for i_trotter in range(n_trotter):
                u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
                u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
            # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)
            # u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
            # for i_trotter in range(n_trotter-1):
            #     u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
            #     u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
            # u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt/2.0)@u_trotter
        elif (indx==1):
            u_trotter = np.identity((dim_sub[isec]),dtype=complex)
            for i_trotter in range(n_trotter):
                u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
                u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
            #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)
            #u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
            #for i_trotter in range(n_trotter-1):
            #    u_trotter = ExactEvolution_coulomb (isec, 0.0, dt)@u_trotter
            #    u_trotter = ExactEvolution_kinetic (isec, alpha, 0.0, dt)@u_trotter
            #u_trotter = ExactEvolution_coulomb (isec, 0.0, dt/2.0)@u_trotter
        return u_trotter*np.exp(1j*eps*time)

    #print('# exact')
    #phi = eigen_vectors_exact[0][0,:,0]
    #for alpha in range(n_hamiltonians):
    #    phi = ExactEvolution(0,alpha,0.0,dt)@phi
    #E = np.real(phi.conj().T@H_subs[0][-1]@phi)
    #fidelity = np.abs(phi.conj().T@eigen_vectors_exact[0][-1,:,0])**2
    ##print(T_asp,E,fidelity)
    #error = np.abs(E-eigen_energies_exact[0][-1,0])
    ##print(T_asp,np.abs(E-eigen_energies_exact[0][-1,0]),fidelity)
    #error_for_dt_list[indx_dt,i_asp] = error
    #print(dt,T_asp,error)

    def NumberOfTrotterSteps(alpha, max_n_trotter):
        trotter_factor = (n_dimer + t_inters[alpha]/t_intra * n_inter)/(n_dimer + n_inter)
        return max(1,int(trotter_factor*max_n_trotter))

    trotter_permutation_indx = 1
    print('# trotter')
    phi = eigen_vectors_exact[0][0,:,0]
    n_trotter_sum = 0
    for alpha in range(n_hamiltonians):
        n_trotter = NumberOfTrotterSteps(alpha,max_n_trotter)
        n_trotter_sum += n_trotter
        phi = TrotterEvolution(0,alpha,dt,0,n_trotter,trotter_permutation_indx)@phi
    E = np.real(phi.conj().T@H_subs[0][-1]@phi)
    error = np.abs(E-eigen_energies_exact[0][-1,0])
    error_for_trotter_list[indx_max_n_trotter] = error
    trotter_sum_for_trotter_list[indx_max_n_trotter] = n_trotter_sum
    print(n_trotter_sum,error)
    indx_max_n_trotter += 1











# trotter
22 2.1965581310162348
# trotter
36 1.5704727238441532
# trotter
54 0.5553231173733524
# trotter
69 0.3810369390661883
# trotter
85 0.09772495236187417
# trotter
100 0.138194235242465
# trotter
118 0.08764493753080771
# trotter
131 0.09012479663610229
# trotter
150 0.05337567737957993
# trotter
310 0.012714747513939528
# trotter
465 0.006207073057666435
# trotter
630 0.00441913808460459
# trotter
780 0.003495074388652597
# trotter
940 0.0029673684678455103


In [53]:
# save to files
with open('dE_vs_n_trotter.save','w') as file_:
    s = '# T_asp= '+str(T_asp) + 'dt= '+str(dt) + 'for n_trotter'
    s += '\n'
    file_.write(s)
    for indx_max_n_trotter in range(n_max_n_trotter):
        s = '{:}'.format(trotter_sum_for_trotter_list[indx_max_n_trotter])
        s += '  {:.16e}'.format(error_for_trotter_list[indx_max_n_trotter])
        print(s)
        s += '\n'
        file_.write(s)

22.0  2.1965581310162348e+00
36.0  1.5704727238441532e+00
54.0  5.5532311737335238e-01
69.0  3.8103693906618830e-01
85.0  9.7724952361874173e-02
100.0  1.3819423524246499e-01
118.0  8.7644937530807709e-02
131.0  9.0124796636102289e-02
150.0  5.3375677379579933e-02
310.0  1.2714747513939528e-02
465.0  6.2070730576664346e-03
630.0  4.4191380846045902e-03
780.0  3.4950743886525970e-03
940.0  2.9673684678455103e-03


In [ ]:
def ExactAdiabaticEvolution(time): # n_asp = n_hamiltonians
    phi = eigen_vectors_exact[0][0,:,0]
    dtime = time/(n_hamiltonians-1)
    for alpha in range(1,n_hamiltonians):
        phi = ExactEvolution(0,alpha,0.0,dtime)@phi
    E = np.real(phi.conj().T@H_subs[0][-1]@phi)
#    E = np.real(eigen_vectors_exact[0][-1,:,0].conj().T@H_subs[0][-1]@eigen_vectors_exact[0][-1,:,0])
    fidelity = np.abs(phi.conj().T@eigen_vectors_exact[0][-1,:,0])**2
    return E, fidelity

In [ ]:
time_list = np.linspace(0.0,10*n_hamiltonians,num=2)
for time in time_list:
    e, f = ExactAdiabaticEvolution(time)
    error = np.abs(e - eigen_energies_exact[0][-1,0])
print(time_list[-1], error, f)


120.0 0.002774606444774008 0.9991512056500377


In [ ]:
time = 1.2
isec = 0
phi_exact_1 = ExactEvolution(isec,0,eigen_energies_exact[isec][-1,0],time)@phi_start
phi_exact_2 = ExactEvolution(isec,0,eigen_energies_exact[isec][-1,0],-time)@phi_start
#u_exact_1 = ExactEvolution(isec,0,eigen_energies_exact[isec][-1,0],time)
#u_exact_2 = ExactEvolution(isec,0,eigen_energies_exact[isec][-1,0],-time)
for n_trotter in range(1,12):
    phi_approx_1 = TrotterEvolution(isec,0,time,eigen_energies_exact[isec][-1,0],n_trotter,1)@phi_start
    phi_approx_2 = TrotterEvolution(isec,0,-time,eigen_energies_exact[isec][-1,0],n_trotter,1)@phi_start
    #u_trotter_1 = TrotterEvolution(isec,0,time,eigen_energies_exact[isec][-1,0],n_trotter,1)
    #u_trotter_2 = TrotterEvolution(isec,0,-time,eigen_energies_exact[isec][-1,0],n_trotter,1)
    #print(np.max(np.abs(u_exact_1-u_trotter_1)))
    #print(np.max(np.abs(u_exact_2-u_trotter_2)))

    #u_exact = u_exact_1+u_exact_2
    #u_trotter = u_trotter_1+u_trotter_2
    #print(np.max(np.abs(u_exact-u_trotter)))
    error = np.max(np.abs(phi_exact_1-phi_approx_1))
    print(error*n_trotter)
    #print(np.max(np.abs(phi_exact_1-phi_approx_1)))
    #print('')
    #phi_approx = (phi_approx_1+phi_approx_2)/2
    #phi_exact = (phi_exact_1+phi_exact_2)/2
    #print(np.max(np.abs(phi_exact-phi_approx)))
    #print('--')
    #o_exact = phi_start@phi_exact
    #o_approx = phi_start@phi_approx
    #print(np.max(np.abs(o_exact-o_approx)))


0.48384640771577797
0.38633735135859804
0.34520998084493554
0.2855621355313122
0.2434968021396008
0.21424710089752338
0.19556218857904992
0.19480686252781754
0.19427673122676736
0.19387906588028714
0.19356690134490173


In [ ]:
n_shot = 2048
nmc    = 16384
n_iter = 20
# beta parameter list
# N_alpha = 4 is fixed
#beta_list = [0.1, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0]
#beta_list = [0.1, 0.5, 1.0, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0, 4.0]
beta_list = np.linspace(0.1,4.0,num=51)
#beta_list = [2.0]
max_evolution_beta_list = np.zeros(len(beta_list),dtype=float)
error_for_beta_list     = np.zeros(len(beta_list),dtype=float)


In [ ]:
# fix random seed for now
#np.random.seed(42)

In [ ]:
# from exact gaussian. no iteration is needed
isec = 0
indx_beta =0
# construct sub matrix
H_subs = [0 for _ in range(n_hamiltonians)]
for alpha in range(n_hamiltonians):
    H_sparse = hamiltonians[alpha].to_matrix(sparse=True)
    H_sparse.eliminate_zeros()
    jsec = isec
    row      = []
    col      = []
    data     = []
    for ii in range(dim_sub[isec]):
        i = indx_sub[isec][ii]
        for ind in range(H_sparse.indptr[i],H_sparse.indptr[i+1]):
            # j is always in indx_sub[isec], because the Hamiltonian does not mix it
            #print(i,j)
            j = H_sparse.indices[ind]
            jj = iindx_sub[jsec][j]
            row.append(jj)
            col.append(ii)
            #print(ii,jj)
            data.append(H_sparse.data[ind])
    H_sub = sparse.csc_matrix((data, (row, col)), shape=(dim_sub[jsec], dim_sub[isec]))
    H_subs[alpha] = H_sub.toarray()
for beta in beta_list:

    eps = eigen_energies_exact[0][0,0]
    norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
    eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)
    eigen_energies_qzmc[0] = eigen_energies_exact[0][0,0]
    phi = copy.copy(phi_start)
    phi = ExactGaussian(isec,0,eigen_energies_qzmc[0],beta)@phi
    norms_qzmc[0] = np.real(phi.conj().T@phi)
    for alpha in range(1,n_hamiltonians):
        phi_1 = (H_subs[alpha]-H_subs[alpha-1])@phi
        phi_1 = ExactGaussian(isec,alpha,eps,beta)@phi_1
        phi = ExactGaussian(isec,alpha,eps,beta)@phi

        norms_qzmc[alpha] = np.real(phi.conj().T@phi)
        dE1 = np.real(phi.conj().T@phi_1)/norms_qzmc[alpha]
        eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
        if (alpha<n_hamiltonians-1):
            phi_2 = (H_subs[alpha+1]-H_subs[alpha])@phi
            dE2 = np.real(phi.conj().T@phi_2)/norms_qzmc[alpha]
            eps = eigen_energies_qzmc[alpha] + dE2
        #if (core==0):
        #    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
        #    memory_usage(st)
        #    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[isec][alpha,0])
        #    if (alpha<n_hamiltonians-1):
        #        print('precision of the predictor for next', eps-eigen_energies_exact[isec][alpha+1,0])
        #    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
        #    print(st)
    error = np.abs(eigen_energies_qzmc[-1] - eigen_energies_exact[0][-1,0])
    error_for_beta_list[indx_beta] = error
    print(beta,norms_qzmc[-1],error)
    indx_beta += 1
        

0.1 0.6917960960006004 0.22737277373040143
0.178 0.5070998418080266 0.18035819355023186
0.256 0.44837033951124616 0.16006291294359887
0.33399999999999996 0.4340601632515381 0.13267017665152814
0.41200000000000003 0.42765364849143017 0.10940056747748095
0.49 0.4222483779030008 0.09436136563633646
0.568 0.41670378645789535 0.08395613699325999
0.646 0.41082075133120893 0.07543511337994335
0.724 0.4045945374057309 0.06757140513747206
0.8019999999999999 0.39806673894873607 0.05997197714466029
0.88 0.39129257430108827 0.05260708184228946
0.958 0.3843268268489949 0.04556800862455823
1.036 0.3772165033788979 0.03896373272530429
1.114 0.36999823309743224 0.03288471590544795
1.1920000000000002 0.3626982586846772 0.027392895106903126
1.27 0.3553336083060743 0.02252076176316198
1.348 0.3479137816346319 0.018273841481589947
1.4260000000000002 0.34044262562584926 0.014634767468617582
1.504 0.33292019946517437 0.011568153164501993
1.582 0.3253444862975668 0.009025708236098495
1.6600000000000001 0.317

In [ ]:
if (core==0):
    with open('dE_vs_evolution','w') as file_:
        s = '# from exact gaussian'
        s += '\n'
        file_.write(s)
        indx_beta = 0
        for beta in beta_list:
            s = '{:}'.format(beta*C99)
            s += '  {:.16e}'.format(error_for_beta_list[indx_beta])
            print(s)
            s += '\n'
            file_.write(s)
            indx_beta += 1


0.6395983900000001  2.2737277373040143e-01
1.1384851342  1.8035819355023186e-01
1.6373718784  1.6006291294359887e-01
2.1362586225999998  1.3267017665152814e-01
2.6351453668  1.0940056747748095e-01
3.134032111  9.4361365636336458e-02
3.6329188552  8.3956136993259989e-02
4.1318055994  7.5435113379943353e-02
4.6306923436  6.7571405137472063e-02
5.1295790878  5.9971977144660293e-02
5.628465832  5.2607081842289460e-02
6.1273525762  4.5568008624558232e-02
6.6262393204  3.8963732725304290e-02
7.125126064600001  3.2884715905447948e-02
7.624012808800001  2.7392895106903126e-02
8.122899553  2.2520761763161978e-02
8.6217862972  1.8273841481589947e-02
9.120673041400002  1.4634767468617582e-02
9.6195597856  1.1568153164501993e-02
10.1184465298  9.0257082360984953e-03
10.617333274000002  6.9511500880272692e-03
11.1162200182  5.2845752955654746e-03
11.6151067624  3.9660781599923922e-03
12.113993506600002  2.9385213134629495e-03
12.6128802508  2.1494623069608920e-03
13.111766994999998  1.5523129753951

In [ ]:
#indx_beta =0
#for beta in beta_list:
#    error = 0.0
#    max_time = 0.0
#    for i_iter in range(n_iter):
#        # pick timelists
#        n_obs = 3
#        # 0; norm, 1; dE1, 2; dE2
#        O_timelists         = [[[None for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]
#        # %%
#        if (core==0):
#            for alpha in range(1,n_hamiltonians):
#                for i_obs in range(n_obs):
#                    for imc in range(nmc):
#                        times = np.random.normal(0.0, beta, size=2*(alpha+1))
#                        O_timelists[alpha][i_obs][imc] = times
#        O_timelists = comm.bcast(O_timelists,root=0)
#        # max time
#        for alpha in range(1,n_hamiltonians):
#            for i_obs in range(n_obs):
#                for imc in range(nmc):
#                    time_sum = np.sum(np.abs(O_timelists[alpha][i_obs][imc]))
#                    max_time = max(max_time,time_sum)
#
#        eps = eigen_energies_exact[0][0,0]
#        norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
#        eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)
#        eigen_energies_qzmc[0] = eigen_energies_exact[0][0,0]
#
#        for alpha in range(1,n_hamiltonians):
#            start_time = datetime.now()
#                        
#
#            nhd1 = len(hamiltonian_diffs_reduced_list[alpha-1])
#            nhd1_ = nhd1 # no constant contribution
#
#            if (alpha<n_hamiltonians-1):
#                nhd2 = len(hamiltonian_diffs_reduced_list[alpha])
#            else:
#                nhd2 = 0
#            nhd2_ = nhd2 # no constant contribution
#
#            n_pubs = nmc * (1+nhd1_+nhd2_)
#
#            n_pubs_for_ = [0 for _ in range(cores)]
#            remainder         = n_pubs%cores
#            for i_core in range(cores):
#                n_pubs_for_[i_core] = n_pubs//cores
#                if (i_core<remainder):
#                    n_pubs_for_[i_core] += 1
#            #if (core==0 and alpha==1):
#            #    print('# of different quantum circuits to run = ', n_pubs)
#
#            i_start         = sum(n_pubs_for_[:core])
#            i_end           = i_start + n_pubs_for_[core]
#
#            ind_pub = 0
#            ind_pub_core = 0
#
#            result_values_core = [0.0 for _ in range(n_pubs_for_[core])]
#
#            # norm (i_obs==0)
#            i_obs = 0
#            for imc in range(nmc):
#                # check my turn
#                my_turn = ind_pub>=i_start and ind_pub<i_end
#                ind_pub += 1
#                if (not my_turn):
#                    continue
#                # initialize
#                times = O_timelists[alpha][i_obs][imc]
#                i_time = 0
#                phase  = 0.0
#                phi = copy.copy(phi_start)
#                # 
#                for alpha_ in range(alpha):
#                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                    phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                    i_time += 1
#                # P_{\alpha}
#                phase += eps * times[i_time]
#                phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                i_time += 1
#
#                # P_{\alpha}
#                phase += eps * times[i_time]
#                phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                i_time += 1
#
#                for alpha_ in reversed(range(alpha)):
#                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                    phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                    i_time += 1
#
#                amplitude = phi_start.conj().T@phi
#                amplitude *= np.exp(1j*phase)
#                result_values_core[ind_pub_core] = amplitude.real
#                ind_pub_core += 1
#
#            # dE1 (i_obs==1)
#            i_obs = 1
#            for ihd in range(nhd1):
#                for imc in range(nmc):
#                    # check my turn
#                    my_turn = ind_pub>=i_start and ind_pub<i_end
#                    ind_pub += 1
#                    if (not my_turn):
#                        continue
#                    # initialize
#                    times = O_timelists[alpha][i_obs][imc]
#                    i_time = 0
#                    phase  = 0.0
#                    phi = copy.copy(phi_start)
#                    # 
#                    for alpha_ in range(alpha):
#                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                        i_time += 1
#                    # apply pauli
#                    phi = sectored_pauli[0][alpha-1][ihd]@phi
#
#                    # P_{\alpha}
#                    phase += eps * times[i_time]
#                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                    i_time += 1
#
#                    # P_{\alpha}
#                    phase += eps * times[i_time]
#                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                    i_time += 1
#
#                    for alpha_ in reversed(range(alpha)):
#                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                        i_time += 1
#
#                    amplitude = phi_start.conj().T@phi
#                    amplitude *= np.exp(1j*phase)
#                    result_values_core[ind_pub_core] = amplitude.real
#                    ind_pub_core += 1
#
#            # dE2 (i_obs==2)
#            i_obs = 2
#            for ihd in range(nhd1):
#                for imc in range(nmc):
#                    # check my turn
#                    my_turn = ind_pub>=i_start and ind_pub<i_end
#                    ind_pub += 1
#                    if (not my_turn):
#                        continue
#                    # initialize
#                    times = O_timelists[alpha][i_obs][imc]
#                    i_time = 0
#                    phase  = 0.0
#                    phi = copy.copy(phi_start)
#                    # 
#                    for alpha_ in range(alpha):
#                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                        i_time += 1
#                    # P_{\alpha}
#
#                    phase += eps * times[i_time]
#                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                    i_time += 1
#
#                    # apply pauli
#                    phi = sectored_pauli[0][alpha][ihd]@phi
#
#                    # P_{\alpha}
#                    phase += eps * times[i_time]
#                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
#                    i_time += 1
#
#                    for alpha_ in reversed(range(alpha)):
#                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
#                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
#                        i_time += 1
#
#                    amplitude = phi_start.conj().T@phi
#                    amplitude *= np.exp(1j*phase)
#                    result_values_core[ind_pub_core] = amplitude.real
#                    ind_pub_core += 1
#            for i in range(n_pubs_for_[core]):
#                computed_value = result_values_core[i]
#
#                # shot errors
#                p_up = (computed_value + 1.0)/2.0
#                sample = np.random.binomial(n_shot,p_up)
#                shot_error = 2*(sample/(n_shot) - p_up)
#                computed_value += shot_error
#
#                result_values_core[i] = computed_value
#            # bcast
#            #print(result_values_core)
#            comm.Barrier()
#            result_values = []
#            for i_core in range(cores):
#                if (n_pubs_for_[i_core]==0):
#                    continue
#                result_values_temp = comm.bcast(result_values_core,root=i_core)
#                result_values += result_values_temp
#                comm.Barrier()
#
#            # compute energy eigenvalues
#            i_meas = 0
#
#            # 0; norm
#            norm    = 0.0
#            i_obs   = 0
#            for imc in range(nmc):
#                norm   += result_values[i_meas]
#                i_meas += 1
#            # 1; dE1
#            dE1norm = 0.0
#            i_obs   = 1
#            for ihd in range(nhd1):
#                coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
#                for imc in range(nmc):
#                    dE1norm += coeff *result_values[i_meas]
#                    i_meas += 1
#            # 2; dE2
#            dE2norm = 0.0
#            i_obs   = 2
#            for ihd in range(nhd2):
#                coeff = hamiltonian_diffs_reduced_list[alpha][ihd][1]
#                for imc in range(nmc):
#                    dE2norm += coeff *result_values[i_meas]
#                    i_meas += 1
#
#            norm = norm.real
#            dE1norm = dE1norm.real
#            dE2norm = dE2norm.real
#    
#            dE1norm /= norm
#            dE2norm /= norm
#            norm    /= nmc
#
#            eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1norm
#            norms_qzmc[alpha] = norm
#
#            if (alpha<n_hamiltonians-1):
#                eps = eigen_energies_qzmc[alpha] + dE2norm
#                eps = eps.real
#
#            #if (core==0):
#            #    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
#            #    memory_usage(st)
#            #    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[isec][alpha,0])
#            #    if (alpha<n_hamiltonians-1):
#            #        print('precision of the predictor for next', eps-eigen_energies_exact[isec][alpha+1,0])
#            #    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
#            #    print(st)
#        error += np.abs(eigen_energies_qzmc[-1] - eigen_energies_exact[0][-1,0])
#    error /= n_iter
#    error_for_beta_list[indx_beta] = error
#    if (core==0):
#        print(beta*C99, max_time, error)
#    indx_beta += 1
#

0.2 1.0336579987900192 0.22718002224484515
1.0 5.422615530830133 0.09188596926731525
2.0 10.275364617503534 0.0408328410207766
4.0 20.795673922974995 0.007349947465259632
4.4 22.319810915709848 0.004122274702673368
4.8 25.87076864963916 0.007570849226817789
5.2 28.97300187439834 0.00890693600094843
5.6 29.40351832586194 0.005977296059876114
6.0 30.501482084520426 0.0084600881396641
8.0 44.29524887584289 0.010639675264322569


In [ ]:
#if (core==0):
#    with open('dE_vs_evolution','w') as file_:
#        s = '# nmc= '+str(nmc)+', n_shot= '+str(n_shot)+', n_iter= '+str(n_iter)
#        s += '\n'
#        file_.write(s)
#        indx_beta = 0
#        for beta in beta_list:
#            s = '{:}'.format(beta*(n_hamiltonians))
#            s += '  {:.16e}'.format(error_for_beta_list[indx_beta])
#            print(s)
#            s += '\n'
#            file_.write(s)
#            indx_beta += 1
#

0.2  2.2737277373040143e-01
1.0  9.2844754389182604e-02
2.0  4.1951450844046612e-02
4.0  1.9152551035208631e-03
4.4  8.0092849872048788e-04
4.8  3.0808285310346406e-04
5.2  1.0903176027454720e-04
5.6  3.5506192646472812e-05
6.0  1.0640091585578659e-05
8.0  7.3909713904640739e-09


In [ ]:
# repetition test
beta = 1.8
nmcs = [1, 2, 4, 6, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096, 8192, 16384]
error_for_nmcs = np.zeros((len(nmcs)),dtype=float)
n_shot = 2048
n_iter = 30


In [ ]:
# preprocess
indx_nmc = 0
for nmc in nmcs:
    error = 0.0
    max_time = 0.0
    for i_iter in range(n_iter):
        # pick timelists
        n_obs = 3
        # 0; norm, 1; dE1, 2; dE2
        O_timelists         = [[[None for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]
        # %%
        if (core==0):
            for alpha in range(1,n_hamiltonians):
                for i_obs in range(n_obs):
                    for imc in range(nmc):
                        times = np.random.normal(0.0, beta, size=2*(alpha+1))
                        O_timelists[alpha][i_obs][imc] = times
        O_timelists = comm.bcast(O_timelists,root=0)
        # max time
        for alpha in range(1,n_hamiltonians):
            for i_obs in range(n_obs):
                for imc in range(nmc):
                    time_sum = np.sum(np.abs(O_timelists[alpha][i_obs][imc]))
                    max_time = max(max_time,time_sum)

        eps = eigen_energies_exact[0][0,0]
        norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
        eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)
        eigen_energies_qzmc[0] = eigen_energies_exact[0][0,0]

        for alpha in range(1,n_hamiltonians):
            start_time = datetime.now()
                        

            nhd1 = len(hamiltonian_diffs_reduced_list[alpha-1])
            nhd1_ = nhd1 # no constant contribution

            if (alpha<n_hamiltonians-1):
                nhd2 = len(hamiltonian_diffs_reduced_list[alpha])
            else:
                nhd2 = 0
            nhd2_ = nhd2 # no constant contribution

            n_pubs = nmc * (1+nhd1_+nhd2_)

            n_pubs_for_ = [0 for _ in range(cores)]
            remainder         = n_pubs%cores
            for i_core in range(cores):
                n_pubs_for_[i_core] = n_pubs//cores
                if (i_core<remainder):
                    n_pubs_for_[i_core] += 1
            #if (core==0 and alpha==1):
            #    print('# of different quantum circuits to run = ', n_pubs)

            i_start         = sum(n_pubs_for_[:core])
            i_end           = i_start + n_pubs_for_[core]

            ind_pub = 0
            ind_pub_core = 0

            result_values_core = [0.0 for _ in range(n_pubs_for_[core])]

            # norm (i_obs==0)
            i_obs = 0
            for imc in range(nmc):
                # check my turn
                my_turn = ind_pub>=i_start and ind_pub<i_end
                ind_pub += 1
                if (not my_turn):
                    continue
                # initialize
                times = O_timelists[alpha][i_obs][imc]
                i_time = 0
                phase  = 0.0
                phi = copy.copy(phi_start)
                # 
                for alpha_ in range(alpha):
                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
                    phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                    i_time += 1
                # P_{\alpha}
                phase += eps * times[i_time]
                phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                i_time += 1

                # P_{\alpha}
                phase += eps * times[i_time]
                phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                i_time += 1

                for alpha_ in reversed(range(alpha)):
                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
                    phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                    i_time += 1

                amplitude = phi_start.conj().T@phi
                amplitude *= np.exp(1j*phase)
                result_values_core[ind_pub_core] = amplitude.real
                ind_pub_core += 1

            # dE1 (i_obs==1)
            i_obs = 1
            for ihd in range(nhd1):
                for imc in range(nmc):
                    # check my turn
                    my_turn = ind_pub>=i_start and ind_pub<i_end
                    ind_pub += 1
                    if (not my_turn):
                        continue
                    # initialize
                    times = O_timelists[alpha][i_obs][imc]
                    i_time = 0
                    phase  = 0.0
                    phi = copy.copy(phi_start)
                    # 
                    for alpha_ in range(alpha):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                        i_time += 1
                    # apply pauli
                    phi = sectored_pauli[0][alpha-1][ihd]@phi

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                    i_time += 1

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                    i_time += 1

                    for alpha_ in reversed(range(alpha)):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                        i_time += 1

                    amplitude = phi_start.conj().T@phi
                    amplitude *= np.exp(1j*phase)
                    result_values_core[ind_pub_core] = amplitude.real
                    ind_pub_core += 1

            # dE2 (i_obs==2)
            i_obs = 2
            for ihd in range(nhd1):
                for imc in range(nmc):
                    # check my turn
                    my_turn = ind_pub>=i_start and ind_pub<i_end
                    ind_pub += 1
                    if (not my_turn):
                        continue
                    # initialize
                    times = O_timelists[alpha][i_obs][imc]
                    i_time = 0
                    phase  = 0.0
                    phi = copy.copy(phi_start)
                    # 
                    for alpha_ in range(alpha):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                        i_time += 1
                    # P_{\alpha}

                    phase += eps * times[i_time]
                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                    i_time += 1

                    # apply pauli
                    phi = sectored_pauli[0][alpha][ihd]@phi

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = ExactEvolution(isec,alpha,0.0,times[i_time])@phi
                    i_time += 1

                    for alpha_ in reversed(range(alpha)):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = ExactEvolution(isec,alpha_,0.0,times[i_time])@phi
                        i_time += 1

                    amplitude = phi_start.conj().T@phi
                    amplitude *= np.exp(1j*phase)
                    result_values_core[ind_pub_core] = amplitude.real
                    ind_pub_core += 1
            for i in range(n_pubs_for_[core]):
                computed_value = result_values_core[i]

                # shot errors
                p_up = (computed_value + 1.0)/2.0
                sample = np.random.binomial(n_shot,p_up)
                shot_error = 2*(sample/(n_shot) - p_up)
                computed_value += shot_error

                result_values_core[i] = computed_value
            # bcast
            #print(result_values_core)
            comm.Barrier()
            result_values = []
            for i_core in range(cores):
                if (n_pubs_for_[i_core]==0):
                    continue
                result_values_temp = comm.bcast(result_values_core,root=i_core)
                result_values += result_values_temp
                comm.Barrier()

            # compute energy eigenvalues
            i_meas = 0

            # 0; norm
            norm    = 0.0
            i_obs   = 0
            for imc in range(nmc):
                norm   += result_values[i_meas]
                i_meas += 1
            # 1; dE1
            dE1norm = 0.0
            i_obs   = 1
            for ihd in range(nhd1):
                coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
                for imc in range(nmc):
                    dE1norm += coeff *result_values[i_meas]
                    i_meas += 1
            # 2; dE2
            dE2norm = 0.0
            i_obs   = 2
            for ihd in range(nhd2):
                coeff = hamiltonian_diffs_reduced_list[alpha][ihd][1]
                for imc in range(nmc):
                    dE2norm += coeff *result_values[i_meas]
                    i_meas += 1

            norm = norm.real
            dE1norm = dE1norm.real
            dE2norm = dE2norm.real
    
            dE1norm /= norm
            dE2norm /= norm
            norm    /= nmc

            eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1norm
            norms_qzmc[alpha] = norm

            if (alpha<n_hamiltonians-1):
                eps = eigen_energies_qzmc[alpha] + dE2norm
                eps = eps.real

            #if (core==0):
            #    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
            #    memory_usage(st)
            #    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[isec][alpha,0])
            #    if (alpha<n_hamiltonians-1):
            #        print('precision of the predictor for next', eps-eigen_energies_exact[isec][alpha+1,0])
            #    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
            #    print(st)
        error += np.abs(eigen_energies_qzmc[-1] - eigen_energies_exact[0][-1,0])
    error /= n_iter
    error_for_nmcs[indx_nmc] = error
    if (core==0):
        print(n_pubs, max_time, error)
    indx_nmc += 1

2 12.436116280471746 0.739931424161441
4 11.715697458771523 0.6748822284724406
8 14.837968683667105 0.36940137144142593
12 12.052212181879632 0.27577661587495045
16 12.90943176286734 0.2760631990349324
32 16.034286294555212 0.15392736459554116
64 14.366178059643973 0.09217392087356932
128 15.589583645962858 0.08645282320372309
256 16.466488491791587 0.04505146767654544
512 17.09105374421896 0.03372283341925059
1024 17.012742862448235 0.025910210879035844
2048 17.351628824365918 0.01891410464061186
4096 18.45971033477986 0.012785253196793888
8192 18.495785350421315 0.00848332777652221
16384 18.541878696424064 0.0067094334511595
32768 19.378819096075603 0.005360848486079502


In [ ]:
if (core==0):
    with open('dE_vs_repetitions_with_exact','w') as file_:
        s = '# beta = '+str(beta)+', n_shot= '+str(n_shot)+', n_iter='+str(n_iter)
        s += '\n'
        file_.write(s)
        indx_nmc = 0
        for nmc in nmcs:
            s = '{:}'.format(nmc*(1+nhd1_))
            s += '  {:.16e}'.format(error_for_nmcs[indx_nmc])
            print(s)
            s += '\n'
            file_.write(s)
            indx_nmc += 1


2  7.3993142416144098e-01
4  6.7488222847244062e-01
8  3.6940137144142593e-01
12  2.7577661587495045e-01
16  2.7606319903493243e-01
32  1.5392736459554116e-01
64  9.2173920873569321e-02
128  8.6452823203723092e-02
256  4.5051467676545442e-02
512  3.3722833419250593e-02
1024  2.5910210879035844e-02
2048  1.8914104640611860e-02
4096  1.2785253196793888e-02
8192  8.4833277765222100e-03
16384  6.7094334511595003e-03
32768  5.3608484860795018e-03


In [ ]:
# for trotter
# fix time_max
#time_max = time_max_list[5]
beta = 1.8
nmc  = 16384
max_n_trotter_list = [2, 4, 6, 8, 10, 12, 16, 20, 30, 40, 50]
error_for_trotter_list = np.zeros((len(max_n_trotter_list)),dtype=float)
max_n_trotter_run_list = np.zeros((len(max_n_trotter_list)),dtype=float)
n_shot = 2048
n_iter = 30
trotter_permutation_indx     = 1
trotter_permutation_indx_dag = 1 # fixed to 1 for a fair comparision


In [ ]:
def NumberOfTrotterSteps(alpha, max_n_trotter):
    trotter_factor = (n_dimer + t_inters[alpha]/t_intra * n_inter)/(n_dimer + n_inter)
    return max(1,int(trotter_factor*max_n_trotter))

In [ ]:
# trotterized case
indx_max_n_trotter = 0
for max_n_trotter in max_n_trotter_list:
    start_time = datetime.now()

    n_trotters = [NumberOfTrotterSteps(alpha, max_n_trotter) for alpha in range(n_hamiltonians)]

    error = 0.0
    max_time = 0.0
    for i_iter in range(n_iter):
        # pick timelists
        n_obs = 3
        # 0; norm, 1; dE1, 2; dE2
        O_timelists         = [[[None for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]
        # %%
        if (core==0):
            for alpha in range(1,n_hamiltonians):
                for i_obs in range(n_obs):
                    for imc in range(nmc):
                        times = np.random.normal(0.0, beta, size=2*(alpha+1))
                        O_timelists[alpha][i_obs][imc] = times
        O_timelists = comm.bcast(O_timelists,root=0)
        # max time
        for alpha in range(1,n_hamiltonians):
            for i_obs in range(n_obs):
                for imc in range(nmc):
                    time_sum = np.sum(np.abs(O_timelists[alpha][i_obs][imc]))
                    max_time = max(max_time,time_sum)

        eps = eigen_energies_exact[0][0,0]
        norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
        eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)
        eigen_energies_qzmc[0] = eigen_energies_exact[0][0,0]

        for alpha in range(1,n_hamiltonians):
            start_time = datetime.now()
                        

            nhd1 = len(hamiltonian_diffs_reduced_list[alpha-1])
            nhd1_ = nhd1 # no constant contribution

            if (alpha<n_hamiltonians-1):
                nhd2 = len(hamiltonian_diffs_reduced_list[alpha])
            else:
                nhd2 = 0
            nhd2_ = nhd2 # no constant contribution

            n_pubs = nmc * (1+nhd1_+nhd2_)

            n_pubs_for_ = [0 for _ in range(cores)]
            remainder         = n_pubs%cores
            for i_core in range(cores):
                n_pubs_for_[i_core] = n_pubs//cores
                if (i_core<remainder):
                    n_pubs_for_[i_core] += 1
            #if (core==0 and alpha==1):
            #    print('# of different quantum circuits to run = ', n_pubs)

            i_start         = sum(n_pubs_for_[:core])
            i_end           = i_start + n_pubs_for_[core]

            ind_pub = 0
            ind_pub_core = 0

            result_values_core = [0.0 for _ in range(n_pubs_for_[core])]

            # norm (i_obs==0)
            i_obs = 0
            for imc in range(nmc):
                # check my turn
                my_turn = ind_pub>=i_start and ind_pub<i_end
                ind_pub += 1
                if (not my_turn):
                    continue
                # initialize
                times = O_timelists[alpha][i_obs][imc]
                i_time = 0
                phase  = 0.0
                phi = copy.copy(phi_start)
                # 
                for alpha_ in range(alpha):
                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
                    phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx)@phi
                    i_time += 1
                # P_{\alpha}
                phase += eps * times[i_time]
                phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx)@phi
                i_time += 1

                # P_{\alpha}
                phase += eps * times[i_time]
                phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx_dag)@phi
                i_time += 1

                for alpha_ in reversed(range(alpha)):
                    phase += eigen_energies_qzmc[alpha_] * times[i_time]
                    phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx_dag)@phi
                    i_time += 1

                amplitude = phi_start.conj().T@phi
                amplitude *= np.exp(1j*phase)
                result_values_core[ind_pub_core] = amplitude.real
                ind_pub_core += 1

            # dE1 (i_obs==1)
            i_obs = 1
            for ihd in range(nhd1):
                for imc in range(nmc):
                    # check my turn
                    my_turn = ind_pub>=i_start and ind_pub<i_end
                    ind_pub += 1
                    if (not my_turn):
                        continue
                    # initialize
                    times = O_timelists[alpha][i_obs][imc]
                    i_time = 0
                    phase  = 0.0
                    phi = copy.copy(phi_start)
                    # 
                    for alpha_ in range(alpha):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx)@phi
                        i_time += 1
                    # apply pauli
                    phi = sectored_pauli[0][alpha-1][ihd]@phi

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx)@phi
                    i_time += 1

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx_dag)@phi
                    i_time += 1

                    for alpha_ in reversed(range(alpha)):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx_dag)@phi
                        i_time += 1

                    amplitude = phi_start.conj().T@phi
                    amplitude *= np.exp(1j*phase)
                    result_values_core[ind_pub_core] = amplitude.real
                    ind_pub_core += 1

            # dE2 (i_obs==2)
            i_obs = 2
            for ihd in range(nhd1):
                for imc in range(nmc):
                    # check my turn
                    my_turn = ind_pub>=i_start and ind_pub<i_end
                    ind_pub += 1
                    if (not my_turn):
                        continue
                    # initialize
                    times = O_timelists[alpha][i_obs][imc]
                    i_time = 0
                    phase  = 0.0
                    phi = copy.copy(phi_start)
                    # 
                    for alpha_ in range(alpha):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx)@phi
                        i_time += 1
                    # P_{\alpha}

                    phase += eps * times[i_time]
                    phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx)@phi
                    i_time += 1

                    # apply pauli
                    phi = sectored_pauli[0][alpha][ihd]@phi

                    # P_{\alpha}
                    phase += eps * times[i_time]
                    phi = TrotterEvolution(isec, alpha, times[i_time], 0.0, n_trotters[alpha], trotter_permutation_indx_dag)@phi
                    i_time += 1

                    for alpha_ in reversed(range(alpha)):
                        phase += eigen_energies_qzmc[alpha_] * times[i_time]
                        phi = TrotterEvolution(isec, alpha_, times[i_time], 0.0, n_trotters[alpha_], trotter_permutation_indx_dag)@phi
                        i_time += 1

                    amplitude = phi_start.conj().T@phi
                    amplitude *= np.exp(1j*phase)
                    result_values_core[ind_pub_core] = amplitude.real
                    ind_pub_core += 1
            for i in range(n_pubs_for_[core]):
                computed_value = result_values_core[i]

                # shot errors
                p_up = (computed_value + 1.0)/2.0
                sample = np.random.binomial(n_shot,p_up)
                shot_error = 2*(sample/(n_shot) - p_up)
                computed_value += shot_error

                result_values_core[i] = computed_value
            # bcast
            #print(result_values_core)
            comm.Barrier()
            result_values = []
            for i_core in range(cores):
                if (n_pubs_for_[i_core]==0):
                    continue
                result_values_temp = comm.bcast(result_values_core,root=i_core)
                result_values += result_values_temp
                comm.Barrier()

            # compute energy eigenvalues
            i_meas = 0

            # 0; norm
            norm    = 0.0
            i_obs   = 0
            for imc in range(nmc):
                norm   += result_values[i_meas]
                i_meas += 1
            # 1; dE1
            dE1norm = 0.0
            i_obs   = 1
            for ihd in range(nhd1):
                coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
                for imc in range(nmc):
                    dE1norm += coeff *result_values[i_meas]
                    i_meas += 1
            # 2; dE2
            dE2norm = 0.0
            i_obs   = 2
            for ihd in range(nhd2):
                coeff = hamiltonian_diffs_reduced_list[alpha][ihd][1]
                for imc in range(nmc):
                    dE2norm += coeff *result_values[i_meas]
                    i_meas += 1

            norm = norm.real
            dE1norm = dE1norm.real
            dE2norm = dE2norm.real
    
            dE1norm /= norm
            dE2norm /= norm
            norm    /= nmc

            eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1norm
            norms_qzmc[alpha] = norm

            if (alpha<n_hamiltonians-1):
                eps = eigen_energies_qzmc[alpha] + dE2norm
                eps = eps.real

            #if (core==0):
            #    st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
            #    memory_usage(st)
            #    print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[isec][alpha,0])
            #    if (alpha<n_hamiltonians-1):
            #        print('precision of the predictor for next', eps-eigen_energies_exact[isec][alpha+1,0])
            #    st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
            #    print(st)
        error += np.abs(eigen_energies_qzmc[-1] - eigen_energies_exact[0][-1,0])
    error /= n_iter
    error /= n_iter
    error_for_trotter_list[indx_max_n_trotter] = error
    max_n_trotter_run_list[indx_max_n_trotter] = 2*np.sum(n_trotters)
    if (core==0):
        print(2*np.sum(n_trotters), error)
    indx_max_n_trotter += 1
    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    if (core==0):
        st = '# elapsed time = {elapsed} secs'.format(elapsed=elapsed)
        memory_usage(st)



36 0.013314635011276943
[# elapsed time = 64.885573 secs] memory usage:    0.28852 GiB


In [ ]:
if (core==0):
    with open('dE_vs_n_trotter','w') as file_:
        s = '# beta= '+str(beta)+', n_shot= '+str(n_shot)+', n_iter= '+str(n_iter)+', nmc= '+str(nmc)
        s += '\n'
        file_.write(s)
        indx_n_trotter = 0
        for max_n_trotter in max_n_trotter_list:
            s = '{:}'.format(max_n_trotter_run_list[indx_n_trotter])
            s += '  {:.16e}'.format(error_for_trotter_list[indx_n_trotter])
            print(s)
            s += '\n'
            file_.write(s)
            indx_n_trotter += 1

30.0  5.6627778941775730e-03
